# 🦠 Finder Agenti Patogeni

**Problema:** Vreau sa construiesc un sistem care primeste numele unei boli si returneaza agentii patogeni asociati cu frecventele lor.

**Contextul:** Medicii sau cercetatorii pot cauta rapid ce agenti patogeni sunt asociati cu o anumita boala, fara sa caute manual prin mii de randuri.

**Impactul:** Sistemul ajuta la identificarea rapida a agentilor patogeni si frecventele lor, utile in cercetarea medicala.

**Dataset:** vega_pathogens_catalog.csv - catalog cu agenti patogeni, frecvente si boli asociate
- Coloane: pathogen_name, pathogen_type, frequency_hz, disease_count, associated_diseases, target_organs
- Peste 700 de intrari cu agenti patogeni diferiti

## Pasul 1 - Instalam ce avem nevoie

In [ ]:
# instalam librariile necesare
!pip install pydantic sentence-transformers scikit-learn pandas matplotlib seaborn

## Pasul 2 - Cream folderele si importam librariile

In [ ]:
# cream folderele necesare
import os
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)
print('folderele au fost create!')

In [ ]:
# importam tot ce avem nevoie
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pickle
import warnings
warnings.filterwarnings('ignore')

# pydantic - pentru validarea datelor
# ne asigura ca datele au formatul corect
from pydantic import BaseModel, Field
from typing import List, Optional

# pentru cautarea semantica
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

print('toate librariile sunt incarcate!')

## Pasul 3 - Definim structura datelor cu Pydantic

Pydantic ne ajuta sa definim cum trebuie sa arate datele noastre. E ca un formular cu campuri obligatorii.

In [ ]:
# definim structura unui agent patogen cu Pydantic
# asta inseamna ca fiecare agent patogen TREBUIE sa aiba aceste campuri

class AgentPatogen(BaseModel):
    """
    Aceasta clasa defineste cum arata un agent patogen in sistemul nostru.
    Pydantic verifica automat ca toate campurile sunt completate corect.
    """
    nume: str = Field(description="Numele agentului patogen")
    tip: str = Field(description="Tipul: Bacterie, Parazit, Virus etc")
    frecventa: str = Field(description="Frecventa in Hz")
    boli_asociate: List[str] = Field(description="Lista bolilor asociate")
    organe_tinta: List[str] = Field(description="Lista organelor afectate")
    numar_boli: int = Field(description="Numarul de boli asociate")


class RezultatCautare(BaseModel):
    """
    Aceasta clasa defineste cum arata un rezultat al cautarii.
    """
    boala_cautata: str = Field(description="Boala pe care am cautat-o")
    agenti_gasiti: List[AgentPatogen] = Field(description="Lista agentilor patogeni gasiti")
    numar_rezultate: int = Field(description="Cati agenti au fost gasiti")
    mesaj: str = Field(description="Un mesaj descriptiv despre rezultat")


print('Structurile Pydantic au fost definite!')
print()
print('AgentPatogen are campurile:')
for camp, info in AgentPatogen.model_fields.items():
    print(f'  - {camp}: {info.description}')

## Pasul 4 - Incarcam si curatam datele

In [ ]:
# incarcam fisierul CSV cu agentii patogeni
# IMPORTANT: fisierul trebuie sa fie in folderul data/

try:
    df = pd.read_csv('data/vega_pathogens_catalog.csv')
    print('Fisierul a fost incarcat cu succes!')
except:
    # daca nu gaseste fisierul in data/, incearca in folderul curent
    df = pd.read_csv('vega_pathogens_catalog.csv')
    print('Fisierul a fost incarcat din folderul curent!')

print(f'Avem {len(df)} randuri si {len(df.columns)} coloane')
print(f'Coloanele sunt: {list(df.columns)}')

In [ ]:
# vedem primele randuri
print('=== PRIMELE 5 RANDURI ===')
df.head()

In [ ]:
# informatii generale
print('=== INFORMATII GENERALE ===')
df.info()

In [ ]:
# verificam valorile lipsa
print('=== VALORI LIPSA ===')
print(df.isnull().sum())

In [ ]:
# curatam datele
# completam valorile lipsa cu text descriptiv
df['pathogen_type'] = df['pathogen_type'].fillna('Necunoscut')
df['frequency_hz'] = df['frequency_hz'].fillna('Necunoscut')
df['associated_diseases'] = df['associated_diseases'].fillna('Necunoscut')
df['target_organs'] = df['target_organs'].fillna('Necunoscut')
df['disease_count'] = df['disease_count'].fillna(0)

# facem toate textele lowercase pentru cautare mai usoara
df['associated_diseases_lower'] = df['associated_diseases'].str.lower()
df['pathogen_name_lower'] = df['pathogen_name'].str.lower()

print('Datele au fost curatate!')
print(f'Avem {df.isnull().sum().sum()} valori lipsa dupa curatare')

## Pasul 5 - EDA (Explorarea Datelor)

Hai sa vedem ce avem in dataset!

In [ ]:
# statistici descriptive
print('=== STATISTICI DESCRIPTIVE ===')
df.describe()

In [ ]:
# GRAFIC 1 - distributia tipurilor de agenti patogeni
plt.figure(figsize=(12, 5))

# numaram cate tipuri avem, dar excludem numerele (care sunt coduri)
tipuri = df['pathogen_type'].value_counts()
# luam doar primele 10 ca sa nu fie prea aglomerat
tipuri_top = tipuri.head(10)

plt.bar(tipuri_top.index, tipuri_top.values, color='steelblue', edgecolor='black')
plt.title('Distributia tipurilor de agenti patogeni', fontsize=14)
plt.xlabel('Tipul agentului patogen')
plt.ylabel('Numar de agenti')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('results/grafic1_tipuri.png')
plt.show()
print('Observatie: Cel mai frecvent tip este Bacterie, urmat de Parazit')

In [ ]:
# GRAFIC 2 - distributia numarului de boli per agent patogen
plt.figure(figsize=(10, 5))

# convertim disease_count la numeric
disease_counts_numeric = pd.to_numeric(df['disease_count'], errors='coerce').dropna()

plt.hist(disease_counts_numeric, bins=20, color='orange', edgecolor='black')
plt.title('Cate boli are fiecare agent patogen', fontsize=14)
plt.xlabel('Numarul de boli asociate')
plt.ylabel('Numar de agenti patogeni')
plt.axvline(disease_counts_numeric.mean(), color='red', linestyle='--',
            label=f'Media: {disease_counts_numeric.mean():.1f}')
plt.legend()
plt.tight_layout()
plt.savefig('results/grafic2_boli_per_agent.png')
plt.show()
print(f'Observatie: In medie, un agent patogen este asociat cu {disease_counts_numeric.mean():.1f} boli')

In [ ]:
# GRAFIC 3 - top 10 agenti patogeni cu cele mai multe boli asociate
plt.figure(figsize=(14, 6))

df['disease_count_numeric'] = pd.to_numeric(df['disease_count'], errors='coerce')
top_agenti = df.nlargest(10, 'disease_count_numeric')[['pathogen_name', 'disease_count_numeric']]

# scurtam numele daca sunt prea lungi
top_agenti['nume_scurt'] = top_agenti['pathogen_name'].str[:30]

plt.barh(top_agenti['nume_scurt'], top_agenti['disease_count_numeric'], color='green', edgecolor='black')
plt.title('Top 10 agenti patogeni cu cele mai multe boli asociate', fontsize=13)
plt.xlabel('Numar de boli asociate')
plt.tight_layout()
plt.savefig('results/grafic3_top_agenti.png')
plt.show()
print('Observatie: Unii agenti patogeni sunt asociati cu zeci de boli diferite')

In [ ]:
# GRAFIC 4 - cele mai frecvente organe afectate
plt.figure(figsize=(14, 6))

# extragem toate organele din coloana target_organs
toate_organele = []
for organe in df['target_organs'].dropna():
    for organ in str(organe).split(';'):
        organ_curat = organ.strip()
        if organ_curat and organ_curat != 'nan':
            toate_organele.append(organ_curat)

from collections import Counter
frecventa_organe = Counter(toate_organele)
top_organe = dict(sorted(frecventa_organe.items(), key=lambda x: x[1], reverse=True)[:15])

plt.barh(list(top_organe.keys()), list(top_organe.values()), color='purple', edgecolor='black')
plt.title('Top 15 organe cel mai des afectate de agentii patogeni', fontsize=13)
plt.xlabel('Numar de agenti patogeni care afecteaza acest organ')
plt.tight_layout()
plt.savefig('results/grafic4_organe.png')
plt.show()
print('Observatie: Sangele (blood) si ficatul (hepar) sunt cele mai frecvent afectate')

In [ ]:
# GRAFIC 5 - numarul de agenti patogeni per tip
plt.figure(figsize=(8, 8))

# grupam tipurile mai putin frecvente in "Altele"
tipuri_principale = ['Bacterie', 'Parazit', 'Necunoscut']
tip_counts = df['pathogen_type'].value_counts()

tip_pie = {}
altele = 0
for tip, count in tip_counts.items():
    if tip in tipuri_principale:
        tip_pie[tip] = count
    else:
        altele += count
if altele > 0:
    tip_pie['Altele/Coduri'] = altele

plt.pie(tip_pie.values(), labels=tip_pie.keys(), autopct='%1.1f%%',
        colors=['steelblue', 'orange', 'green', 'red', 'purple'])
plt.title('Proportia tipurilor de agenti patogeni', fontsize=14)
plt.tight_layout()
plt.savefig('results/grafic5_proportii.png')
plt.show()
print('Observatie: Bacteriile si parazitii reprezinta majoritatea agentilor patogeni')

## Pasul 6 - Preprocesarea datelor pentru cautare

In [ ]:
# pregatim datele pentru sistemul de cautare
# cream o coloana combinata cu toate informatiile despre fiecare agent patogen
# asta ajuta modelul AI sa gaseasca mai bine

def curata_text(text):
    """Curatam textul pentru cautare mai buna"""
    if pd.isna(text):
        return ''
    text = str(text).lower()
    # scoatem caracterele speciale dar pastram literele romanesti
    text = re.sub(r'[^\w\săîâșțĂÎÂȘȚ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# cream coloana combinata
df['text_combinat'] = df['associated_diseases'].apply(curata_text)

print('Preprocesarea este gata!')
print()
print('Exemplu de text combinat pentru primul rand:')
print(df['text_combinat'].iloc[0])

## Pasul 7 - Modelul 1: Cautare directa (simpla si rapida)

In [ ]:
# MODEL 1: Cautare directa
# cauta pur si simplu daca cuvantul apare in bolile asociate
# simplu dar eficient

def cauta_direct(boala_cautata, top_n=5):
    """
    Cauta agenti patogeni dupa numele bolii
    Returneaza un obiect Pydantic validat
    """
    boala_lower = boala_cautata.lower().strip()

    # cautam in coloana de boli asociate
    masca = df['associated_diseases_lower'].str.contains(boala_lower, na=False)
    rezultate_df = df[masca].head(top_n)

    # convertim rezultatele in obiecte Pydantic
    agenti_gasiti = []
    for _, rand in rezultate_df.iterrows():
        # lista de boli
        boli = [b.strip() for b in str(rand['associated_diseases']).split(';')]
        # lista de organe
        organe = [o.strip() for o in str(rand['target_organs']).split(';')]

        # cream obiectul Pydantic - el verifica automat ca datele sunt corecte
        agent = AgentPatogen(
            nume=str(rand['pathogen_name']),
            tip=str(rand['pathogen_type']),
            frecventa=str(rand['frequency_hz']),
            boli_asociate=boli[:5],  # maxim 5 boli ca sa nu fie prea lung
            organe_tinta=organe[:5],  # maxim 5 organe
            numar_boli=int(rand['disease_count']) if str(rand['disease_count']).isdigit() else 0
        )
        agenti_gasiti.append(agent)

    # cream rezultatul final validat cu Pydantic
    rezultat = RezultatCautare(
        boala_cautata=boala_cautata,
        agenti_gasiti=agenti_gasiti,
        numar_rezultate=len(agenti_gasiti),
        mesaj=f'Am gasit {len(agenti_gasiti)} agenti patogeni asociati cu "{boala_cautata}"'
    )

    return rezultat


def afiseaza_rezultat(rezultat):
    """Afiseaza rezultatul intr-un format frumos"""
    print(f'\n🔍 {rezultat.mesaj}')
    print('='*60)

    if rezultat.numar_rezultate == 0:
        print('Nu am gasit niciun agent patogen pentru aceasta boala.')
        print('Incearca cu alt cuvant cheie!')
        return

    for i, agent in enumerate(rezultat.agenti_gasiti, 1):
        print(f'\n{i}. 🦠 {agent.nume}')
        print(f'   Tip: {agent.tip}')
        print(f'   Frecventa: {agent.frecventa} Hz')
        print(f'   Numar boli asociate: {agent.numar_boli}')
        print(f'   Boli principale: {" | ".join(agent.boli_asociate[:3])}')
        print(f'   Organe afectate: {" | ".join(agent.organe_tinta[:3])}')


# testam
rezultat = cauta_direct('ACNEE')
afiseaza_rezultat(rezultat)

## Pasul 8 - Modelul 2: Cautare cu TF-IDF (mai destept)

In [ ]:
# MODEL 2: TF-IDF
# mai bun decat cautarea directa
# gaseste si rezultate partiale

print('Antrenam modelul TF-IDF...')

tfidf = TfidfVectorizer(ngram_range=(1, 2))  # ngram_range=(1,2) inseamna ca ia si perechi de cuvinte
matrice_tfidf = tfidf.fit_transform(df['text_combinat'])

print(f'TF-IDF a invatat {len(tfidf.vocabulary_)} termeni!')

# salvam modelul
with open('models/model_tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print('Modelul TF-IDF salvat!')


def cauta_cu_tfidf(boala_cautata, top_n=5):
    """Cauta cu TF-IDF si returneaza rezultat Pydantic"""
    boala_curata = curata_text(boala_cautata)
    vector_cautare = tfidf.transform([boala_curata])
    scoruri = cosine_similarity(vector_cautare, matrice_tfidf)[0]

    # luam doar rezultatele cu scor mai mare de 0 (adica relevante)
    indici_relevanti = np.where(scoruri > 0)[0]
    indici_sortati = indici_relevanti[scoruri[indici_relevanti].argsort()[::-1]][:top_n]

    agenti_gasiti = []
    for idx in indici_sortati:
        rand = df.iloc[idx]
        boli = [b.strip() for b in str(rand['associated_diseases']).split(';')]
        organe = [o.strip() for o in str(rand['target_organs']).split(';')]

        agent = AgentPatogen(
            nume=str(rand['pathogen_name']),
            tip=str(rand['pathogen_type']),
            frecventa=str(rand['frequency_hz']),
            boli_asociate=boli[:5],
            organe_tinta=organe[:5],
            numar_boli=int(rand['disease_count']) if str(rand['disease_count']).isdigit() else 0
        )
        agenti_gasiti.append(agent)

    rezultat = RezultatCautare(
        boala_cautata=boala_cautata,
        agenti_gasiti=agenti_gasiti,
        numar_rezultate=len(agenti_gasiti),
        mesaj=f'TF-IDF: Am gasit {len(agenti_gasiti)} agenti pentru "{boala_cautata}"'
    )
    return rezultat


# testam
rezultat_tfidf = cauta_cu_tfidf('sifilis')
afiseaza_rezultat(rezultat_tfidf)

## Pasul 9 - Modelul 3: Cautare Semantica (cel mai bun)

In [ ]:
# MODEL 3: Sentence Transformers
# cel mai bun - intelege sensul cuvintelor
# functioneaza chiar daca scrii putin diferit

print('Se incarca modelul AI semantic... (1-2 minute prima data)')
model_semantic = SentenceTransformer('all-MiniLM-L6-v2')
print('Modelul AI este incarcat!')

print('Se calculeaza amprentele pentru toti agentii patogeni...')
embeddings = model_semantic.encode(df['text_combinat'].tolist(), show_progress_bar=True)
print('Gata!')

# salvam embeddingurile
np.save('models/embeddings.npy', embeddings)
print('Embeddingurile au fost salvate!')


def cauta_semantic(boala_cautata, top_n=5):
    """Cauta semantic si returneaza rezultat Pydantic"""
    emb_cautare = model_semantic.encode([boala_cautata.lower()])
    scoruri = cosine_similarity(emb_cautare, embeddings)[0]
    indici_sortati = scoruri.argsort()[::-1][:top_n]

    # filtram doar cei cu scor relevant (peste 10%)
    indici_relevanti = [idx for idx in indici_sortati if scoruri[idx] > 0.1]

    agenti_gasiti = []
    for idx in indici_relevanti:
        rand = df.iloc[idx]
        boli = [b.strip() for b in str(rand['associated_diseases']).split(';')]
        organe = [o.strip() for o in str(rand['target_organs']).split(';')]

        agent = AgentPatogen(
            nume=str(rand['pathogen_name']),
            tip=str(rand['pathogen_type']),
            frecventa=str(rand['frequency_hz']),
            boli_asociate=boli[:5],
            organe_tinta=organe[:5],
            numar_boli=int(rand['disease_count']) if str(rand['disease_count']).isdigit() else 0
        )
        agenti_gasiti.append(agent)

    rezultat = RezultatCautare(
        boala_cautata=boala_cautata,
        agenti_gasiti=agenti_gasiti,
        numar_rezultate=len(agenti_gasiti),
        mesaj=f'Semantic: Am gasit {len(agenti_gasiti)} agenti pentru "{boala_cautata}"'
    )
    return rezultat


# testam
rezultat_sem = cauta_semantic('toxoplasmoza')
afiseaza_rezultat(rezultat_sem)

## Pasul 10 - Evaluarea si Compararea Modelelor

In [ ]:
# comparam cele 3 modele pe mai multe boli de test
boli_test = ['ACNEE', 'sifilis', 'toxoplasmoza', 'tenie', 'tetanos']

print('COMPARATIE INTRE CELE 3 MODELE')
print('='*70)

rezultate_comparatie = []

for boala in boli_test:
    r1 = cauta_direct(boala)
    r2 = cauta_cu_tfidf(boala)
    r3 = cauta_semantic(boala)

    print(f'\nBoala: "{boala}"')
    print(f'  Cautare Directa : {r1.numar_rezultate} agenti gasiti')
    print(f'  TF-IDF          : {r2.numar_rezultate} agenti gasiti')
    print(f'  Semantic        : {r3.numar_rezultate} agenti gasiti')

    rezultate_comparatie.append({
        'Boala': boala,
        'Cautare Directa': r1.numar_rezultate,
        'TF-IDF': r2.numar_rezultate,
        'Semantic': r3.numar_rezultate
    })

df_comparatie = pd.DataFrame(rezultate_comparatie)
print('\n=== TABEL COMPARATIV ===')
print(df_comparatie.to_string(index=False))

In [ ]:
# GRAFIC 6 - comparatia modelelor
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(boli_test))
width = 0.25

ax.bar(x - width, df_comparatie['Cautare Directa'], width, label='Cautare Directa', color='steelblue')
ax.bar(x, df_comparatie['TF-IDF'], width, label='TF-IDF', color='orange')
ax.bar(x + width, df_comparatie['Semantic'], width, label='Semantic', color='green')

ax.set_xlabel('Boala cautata')
ax.set_ylabel('Numar agenti gasiti')
ax.set_title('Comparatie intre cele 3 modele de cautare', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(boli_test)
ax.legend()
plt.tight_layout()
plt.savefig('results/grafic6_comparatie.png')
plt.show()

print('CONCLUZIE: Cautarea directa este cel mai bun model pentru aceasta problema')
print('pentru ca datele sunt in limba romana si cautam termeni exacti.')
print('Modelul final va folosi cautarea directa ca baza.')

## Pasul 11 - Sistemul Final de Cautare in Chat

In [ ]:
# SISTEMUL FINAL
# combina toate cele 3 modele pentru rezultate maxime

def gaseste_agenti_patogeni(boala_cautata, top_n=5):
    """
    Functia principala a sistemului.
    Primeste numele unei boli si returneaza agentii patogeni.
    Combina toate cele 3 modele.
    Returneaza un obiect Pydantic validat.
    """
    print(f'\n🔍 Se cauta agenti patogeni pentru: "{boala_cautata}"...')

    # incercam intai cautarea directa (cea mai precisa)
    rezultat_direct = cauta_direct(boala_cautata, top_n)

    # daca am gasit destule rezultate, returnam
    if rezultat_direct.numar_rezultate >= 2:
        return rezultat_direct

    # daca nu, incercam TF-IDF
    print('   Cautarea directa nu a gasit suficiente rezultate, incerc TF-IDF...')
    rezultat_tfidf = cauta_cu_tfidf(boala_cautata, top_n)

    if rezultat_tfidf.numar_rezultate >= 2:
        return rezultat_tfidf

    # daca nici asta nu merge, folosim modelul semantic
    print('   Incerc modelul semantic AI...')
    return cauta_semantic(boala_cautata, top_n)


# salvam datasetul procesat
df.to_csv('data/patogeni_procesati.csv', index=False)
print('Datele procesate au fost salvate!')

print()
print('=== SISTEMUL ESTE GATA ===')
print('Poti acum sa cauti orice boala!')

## Pasul 12 - Interfata de Chat

Acesta este locul unde tu scrii numele bolii si primesti raspunsul!

In [ ]:
# ================================================
# INTERFATA DE CHAT - SCRIE BOALA TA AICI!
# ================================================

# Schimba textul de mai jos cu boala pe care o cauti
# Exemple: 'ACNEE', 'sifilis', 'toxoplasmoza', 'tenie', 'tetanos'
# 'boala Lyme', 'leucemie', 'cancer', 'tuberculoza'

BOALA_CAUTATA = 'ACNEE'   # <-- SCHIMBA AICI

# cautam si afisam rezultatul
rezultat_final = gaseste_agenti_patogeni(BOALA_CAUTATA)
afiseaza_rezultat(rezultat_final)

# afisam si in format JSON (structura Pydantic)
print()
print('=== DATE STRUCTURATE (format Pydantic/JSON) ===')
print(rezultat_final.model_dump_json(indent=2)[:1000], '...')

In [ ]:
# mai multe exemple de cautari
print('EXEMPLE DE CAUTARI:')
print()

for boala in ['sifilis', 'toxoplasmoza', 'tetanos']:
    rez = gaseste_agenti_patogeni(boala, top_n=3)
    afiseaza_rezultat(rez)
    print('-'*60)

## Concluzii

**Ce am facut:**
1. Am incarcat un dataset cu peste 700 agenti patogeni
2. Am definit structuri de date cu **Pydantic** pentru validare automata
3. Am explorat datele cu 6 grafice
4. Am curatat si pregatit textul pentru cautare
5. Am antrenat 3 modele: Cautare Directa, TF-IDF, Semantic
6. Am comparat modelele si ales cel mai bun
7. Am creat un sistem final de chat care combina toate modelele

**Ce face Pydantic in proiect:**
- Valideaza automat ca datele sunt corecte
- Asigura ca fiecare agent patogen are toate campurile necesare
- Permite exportul datelor in format JSON

**Ce am observat:**
- Cautarea directa functioneaza cel mai bine pentru termeni exacti in romana
- TF-IDF gaseste si rezultate partiale
- Modelul semantic este util pentru cautari in alte limbi

**Limitari:**
- Dataset-ul contine termeni medicali specializati
- Modelul semantic e antrenat pe engleza, nu pe romana
- Nu este o aplicatie medicala - nu inlocuieste un medic!